In [1]:
!pip install mp-api pymatgen pandas numpy matminer scikit-learn

In [5]:
import os
import pandas as pd
import numpy as np

from getpass import getpass
from mp_api.client import MPRester

In [7]:
import getpass

API_KEY = getpass.getpass(
    "Enter your Materials Project API key: "
)

print("API key loaded successfully.")

Enter your Materials Project API key:  ········


API key loaded successfully.


In [8]:
with MPRester(API_KEY) as mpr:

    docs = mpr.summary.search(
        elements=["O"],
        num_elements=(2, 2),
        is_stable=True,
        fields=[
            "material_id",
            "formula_pretty",
            "formation_energy_per_atom",
            "is_stable"
        ]
    )

print("Total materials retrieved:", len(docs))

C:\Users\adity\AppData\Local\Temp\ipykernel_1396\1325650017.py:3: DeprecationWarning: Accessing summary data through MPRester.summary is deprecated. Please use MPRester.materials.summary instead.
  docs = mpr.summary.search(


Retrieving SummaryDoc documents:   0%|          | 0/166 [00:00<?, ?it/s]

Total materials retrieved: 166


In [9]:
data = []

for doc in docs:

    data.append({
        "material_id": str(doc.material_id),
        "formula_pretty": doc.formula_pretty,
        "formation_energy_per_atom": doc.formation_energy_per_atom
    })

raw_df = pd.DataFrame(data)

print("Dataset shape:", raw_df.shape)

raw_df.head()

Dataset shape: (166, 3)


,material_id,formula_pretty,formation_energy_per_atom
0,mp-2803876,AgO,-0.442572
1,mp-1071163,TiO,-2.907956
2,mp-1095223,OF2,-0.495572
3,mp-1215377,Zr4O,-1.240729
4,mp-1232659,MnO,-2.032570


In [10]:
# task 2 

In [11]:
print("Initial row count:", len(raw_df))

print("\nMissing values:")
print(raw_df.isnull().sum())

print("\nDataset information:")
raw_df.info()

Initial row count: 166

Missing values:
material_id                  0
formula_pretty               0
formation_energy_per_atom    0
dtype: int64

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166 entries, 0 to 165
Data columns (total 3 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   material_id                166 non-null    object 
 1   formula_pretty             166 non-null    object 
 2   formation_energy_per_atom  166 non-null    float64
dtypes: float64(1), object(2)
memory usage: 4.0+ KB


In [12]:
clean_df = raw_df.dropna(
    subset=["formation_energy_per_atom"]
).copy()

print("Rows after removing missing formation energy:",
      len(clean_df))

Rows after removing missing formation energy: 166


In [13]:
clean_df = clean_df.sort_values(
    by="formation_energy_per_atom",
    ascending=True
)

clean_df = clean_df.drop_duplicates(
    subset=["formula_pretty"],
    keep="first"
).reset_index(drop=True)

print("Rows after duplicate removal:", len(clean_df))

clean_df.head()

Rows after duplicate removal: 165


,material_id,formula_pretty,formation_energy_per_atom
0,mp-643,ThO2,-4.367790
1,mp-1427,Lu2O3,-4.123784
2,mp-1767,Tm2O3,-4.090073
3,mp-679,Er2O3,-4.057094
4,mp-2814,Yb2O3,-4.052655


In [14]:
Q1 = clean_df["formation_energy_per_atom"].quantile(0.25)

Q3 = clean_df["formation_energy_per_atom"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR

upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)

print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

Q1: -3.1596579241666647
Q3: -1.2180853000000003
IQR: 1.9415726241666644
Lower Bound: -6.072016860416662
Upper Bound: 1.6942736362499966


In [15]:
clean_df = clean_df[
    (clean_df["formation_energy_per_atom"] >= lower_bound) &
    (clean_df["formation_energy_per_atom"] <= upper_bound)
].reset_index(drop=True)

print("Initial row count:", len(raw_df))

print("Cleaned row count:", len(clean_df))

print("Rows removed:", len(raw_df) - len(clean_df))

clean_df.head()

Initial row count: 166
Cleaned row count: 165
Rows removed: 1


,material_id,formula_pretty,formation_energy_per_atom
0,mp-643,ThO2,-4.367790
1,mp-1427,Lu2O3,-4.123784
2,mp-1767,Tm2O3,-4.090073
3,mp-679,Er2O3,-4.057094
4,mp-2814,Yb2O3,-4.052655


In [16]:
# task -3 

In [17]:
from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementProperty

from sklearn.feature_selection import VarianceThreshold

In [18]:
converter = StrToComposition()

feature_df = converter.featurize_dataframe(
    clean_df.copy(),
    col_id="formula_pretty",
    ignore_errors=True
)

print("Composition conversion completed.")

feature_df[
    ["formula_pretty", "composition"]
].head()

StrToComposition:   0%|          | 0/165 [00:00<?, ?it/s]

Composition conversion completed.


,formula_pretty,composition
0,ThO2,"(Th, O)"
1,Lu2O3,"(Lu, O)"
2,Tm2O3,"(Tm, O)"
3,Er2O3,"(Er, O)"
4,Yb2O3,"(Yb, O)"


In [19]:
ep = ElementProperty.from_preset("magpie")

feature_df = ep.featurize_dataframe(
    feature_df,
    col_id="composition",
    ignore_errors=True
)

print("Magpie features generated successfully.")

print("Total columns:", feature_df.shape[1])

feature_df.head()

ElementProperty:   0%|          | 0/165 [00:00<?, ?it/s]

Magpie features generated successfully.
Total columns: 136


,material_id,formula_pretty,formation_energy_per_atom,composition,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,mp-643,ThO2,-4.367790,"(Th, O)",8.0,90.0,82.0,35.333333,36.444444,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,83.0,94.666667,12.0
1,mp-1427,Lu2O3,-4.123784,"(Lu, O)",8.0,71.0,63.0,33.200000,30.240000,8.0,...,0.002247,0.000899,0.001079,0.0,12.0,194.0,182.0,84.8,87.360000,12.0
2,mp-1767,Tm2O3,-4.090073,"(Tm, O)",8.0,69.0,61.0,32.400000,29.280000,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,194.0,182.0,84.8,87.360000,12.0
3,mp-679,Er2O3,-4.057094,"(Er, O)",8.0,68.0,60.0,32.000000,28.800000,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,194.0,182.0,84.8,87.360000,12.0
4,mp-2814,Yb2O3,-4.052655,"(Yb, O)",8.0,70.0,62.0,32.800000,29.760000,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,97.2,102.240000,12.0


In [20]:
feature_columns = ep.feature_labels()

X = feature_df[feature_columns].copy()

# Convert all features into numeric format
X = X.apply(pd.to_numeric, errors="coerce")

# Replace infinite values with NaN
X = X.replace([np.inf, -np.inf], np.nan)

# Remove columns containing only missing values
X = X.dropna(axis=1, how="all")

# Fill remaining missing values using column median
X = X.fillna(X.median())

print("Feature matrix shape before filtering:", X.shape)

X.head()

Feature matrix shape before filtering: (165, 132)


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,8.0,90.0,82.0,35.333333,36.444444,8.0,16.0,87.0,71.0,63.333333,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,83.0,94.666667,12.0
1,8.0,71.0,63.0,33.200000,30.240000,8.0,41.0,87.0,46.0,68.600000,...,0.002247,0.000899,0.001079,0.0,12.0,194.0,182.0,84.8,87.360000,12.0
2,8.0,69.0,61.0,32.400000,29.280000,8.0,37.0,87.0,50.0,67.000000,...,0.000000,0.000000,0.000000,0.0,12.0,194.0,182.0,84.8,87.360000,12.0
3,8.0,68.0,60.0,32.000000,28.800000,8.0,35.0,87.0,52.0,66.200000,...,0.000000,0.000000,0.000000,0.0,12.0,194.0,182.0,84.8,87.360000,12.0
4,8.0,70.0,62.0,32.800000,29.760000,8.0,39.0,87.0,48.0,67.800000,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,97.2,102.240000,12.0


In [21]:
selector = VarianceThreshold(threshold=0.01)

X_selected = selector.fit_transform(X)

selected_features = X.columns[
    selector.get_support()
]

X_selected_df = pd.DataFrame(
    X_selected,
    columns=selected_features,
    index=X.index
)

print("Feature matrix before filtering:", X.shape)

print("Feature matrix after filtering:", X_selected_df.shape)

Feature matrix before filtering: (165, 132)
Feature matrix after filtering: (165, 120)


In [22]:
print("Selected features:")

for feature in selected_features:
    print(feature)

Selected features:
MagpieData minimum Number
MagpieData maximum Number
MagpieData range Number
MagpieData mean Number
MagpieData avg_dev Number
MagpieData mode Number
MagpieData minimum MendeleevNumber
MagpieData maximum MendeleevNumber
MagpieData range MendeleevNumber
MagpieData mean MendeleevNumber
MagpieData avg_dev MendeleevNumber
MagpieData mode MendeleevNumber
MagpieData minimum AtomicWeight
MagpieData maximum AtomicWeight
MagpieData range AtomicWeight
MagpieData mean AtomicWeight
MagpieData avg_dev AtomicWeight
MagpieData mode AtomicWeight
MagpieData minimum MeltingT
MagpieData maximum MeltingT
MagpieData range MeltingT
MagpieData mean MeltingT
MagpieData avg_dev MeltingT
MagpieData mode MeltingT
MagpieData minimum Column
MagpieData maximum Column
MagpieData range Column
MagpieData mean Column
MagpieData avg_dev Column
MagpieData mode Column
MagpieData maximum Row
MagpieData range Row
MagpieData mean Row
MagpieData avg_dev Row
MagpieData mode Row
MagpieData minimum CovalentRadiu

In [23]:
metadata = feature_df[
    [
        "material_id",
        "formula_pretty",
        "formation_energy_per_atom"
    ]
].reset_index(drop=True)

final_df = pd.concat(
    [
        metadata,
        X_selected_df.reset_index(drop=True)
    ],
    axis=1
)

print("Final featured dataset shape:", final_df.shape)

final_df.head()

Final featured dataset shape: (165, 123)


,material_id,formula_pretty,formation_energy_per_atom,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,...,MagpieData maximum GSmagmom,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,mp-643,ThO2,-4.367790,8.0,90.0,82.0,35.333333,36.444444,8.0,16.0,...,0.000000,0.000000,0.000000,0.000000,12.0,225.0,213.0,83.0,94.666667,12.0
1,mp-1427,Lu2O3,-4.123784,8.0,71.0,63.0,33.200000,30.240000,8.0,41.0,...,0.002247,0.002247,0.000899,0.001079,12.0,194.0,182.0,84.8,87.360000,12.0
2,mp-1767,Tm2O3,-4.090073,8.0,69.0,61.0,32.400000,29.280000,8.0,37.0,...,0.000000,0.000000,0.000000,0.000000,12.0,194.0,182.0,84.8,87.360000,12.0
3,mp-679,Er2O3,-4.057094,8.0,68.0,60.0,32.000000,28.800000,8.0,35.0,...,0.000000,0.000000,0.000000,0.000000,12.0,194.0,182.0,84.8,87.360000,12.0
4,mp-2814,Yb2O3,-4.052655,8.0,70.0,62.0,32.800000,29.760000,8.0,39.0,...,0.000000,0.000000,0.000000,0.000000,12.0,225.0,213.0,97.2,102.240000,12.0


In [24]:
# task  4 

In [25]:
final_df.to_csv(
    "practice_features.csv",
    index=False
)

print("CSV file exported successfully!")

print("Filename: practice_features.csv")

CSV file exported successfully!
Filename: practice_features.csv


In [26]:
import os

if os.path.exists("practice_features.csv"):

    print("File exists!")

    print("File size:",
          os.path.getsize("practice_features.csv"),
          "bytes")

    exported_df = pd.read_csv("practice_features.csv")

    print("\nExported dataset:")
    display(exported_df.head())

else:
    print("CSV export failed.")

File exists!
File size: 148207 bytes

Exported dataset:


,material_id,formula_pretty,formation_energy_per_atom,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,...,MagpieData maximum GSmagmom,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,mp-643,ThO2,-4.367790,8.0,90.0,82.0,35.333333,36.444444,8.0,16.0,...,0.000000,0.000000,0.000000,0.000000,12.0,225.0,213.0,83.0,94.666667,12.0
1,mp-1427,Lu2O3,-4.123784,8.0,71.0,63.0,33.200000,30.240000,8.0,41.0,...,0.002247,0.002247,0.000899,0.001079,12.0,194.0,182.0,84.8,87.360000,12.0
2,mp-1767,Tm2O3,-4.090073,8.0,69.0,61.0,32.400000,29.280000,8.0,37.0,...,0.000000,0.000000,0.000000,0.000000,12.0,194.0,182.0,84.8,87.360000,12.0
3,mp-679,Er2O3,-4.057094,8.0,68.0,60.0,32.000000,28.800000,8.0,35.0,...,0.000000,0.000000,0.000000,0.000000,12.0,194.0,182.0,84.8,87.360000,12.0
4,mp-2814,Yb2O3,-4.052655,8.0,70.0,62.0,32.800000,29.760000,8.0,39.0,...,0.000000,0.000000,0.000000,0.000000,12.0,225.0,213.0,97.2,102.240000,12.0
